In [ ]:
# Stanford in your viewer: fetch real USGS lidar, start the viewer on it, show the page.
# Needs a T4 GPU runtime. Change PLACE below to go somewhere else in the same survey.
import os, shutil, signal, subprocess, sys, time, urllib.request
from IPython.display import Image, display

REPO_URL = "https://github.com/ShreyasxAnand/pointwalk"
BRANCH = "dev"
FLAGS = ["-O2", "-arch=sm_75", "-fmad=false"]
PORT = 8001

SURVEY = "CA_SantaClaraCounty_2020"      # USGS 3DEP survey name in the public lidar bucket
PLACE = dict(lat=37.4275, lon=-122.1700)  # Stanford Main Quad
SIZE_M = 1000      # side of the square to fetch, in metres
DEPTH = 11         # detail: 10 = 8.5 million dots, 11 = 23 million, 12 = several times more
DOT_SIZE_M = 0.3   # how wide each dot is drawn, in metres
FLY_SPEED = 25.0   # metres per second (Shift is 3 times faster)

repo = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
exp = os.path.join(repo, "experiments", "realplace")
build = os.path.join(repo, "build_real")
pid_file = "/content/pointwalk_real.pid"
log_file = "/content/pointwalk_real.log"


def sh(*cmd, cwd=None, quiet=False):
    """Run a command, print its output, stop the cell if it fails."""
    print("$ " + " ".join(cmd), flush=True)
    r = subprocess.run(cmd, cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if not quiet or r.returncode != 0:
        print(r.stdout, flush=True)
    if r.returncode != 0:
        raise SystemExit(f"FAILED (exit {r.returncode}): {' '.join(cmd)}")
    return r.stdout


# 0. Stop the server left over from the last time this cell ran.
if os.path.exists(pid_file):
    try:
        os.killpg(int(open(pid_file).read()), signal.SIGTERM)
        time.sleep(1)
    except (ProcessLookupError, ValueError, PermissionError):
        pass
    os.remove(pid_file)

# 1. Get the code.
if not os.path.isdir(os.path.join(repo, ".git")):
    sh("git", "clone", "--branch", BRANCH, REPO_URL, repo)
else:
    sh("git", "fetch", "origin", BRANCH, cwd=repo)
    sh("git", "checkout", BRANCH, cwd=repo)
    sh("git", "pull", "--ff-only", "origin", BRANCH, cwd=repo)
sh("git", "log", "-1", "--format=%h %ad %s", "--date=iso", cwd=repo)
sh("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")

# 2. Compile the real-data viewer.
os.makedirs(build, exist_ok=True)
sh("nvcc", *FLAGS, "-I", os.path.join(repo, "src"), "-I", exp,
   os.path.join(exp, "viewer_real.cu"), "-o", os.path.join(build, "viewer_real"))

# 3. Fetch the lidar (skipped if this exact request is already on disk).
sh(sys.executable, "-m", "pip", "install", "-q", "laspy[lazrs]", quiet=True)
scene = os.path.join(build, f"{SURVEY}_{PLACE['lat']}_{PLACE['lon']}_{SIZE_M}_{DEPTH}.bin")
if not os.path.exists(scene + ".pose"):
    sh(sys.executable, os.path.join(exp, "fetch_place.py"), "--survey", SURVEY,
       "--lat", str(PLACE["lat"]), "--lon", str(PLACE["lon"]),
       "--size", str(SIZE_M), "--depth", str(DEPTH), "--out", scene)
else:
    print("using the scene already on disk:", scene)

# 4. The page and server are the project's own; only the fly speed is changed, in a copy.
page = open(os.path.join(repo, "viewer", "index.html")).read()
faster = page.replace("const WALK_SPEED = 3.0;", f"const WALK_SPEED = {FLY_SPEED};")
if faster == page:
    print("note: could not change the speed in the page; it will move at walking speed")
open(os.path.join(build, "index.html"), "w").write(faster)
shutil.copy(os.path.join(repo, "viewer", "server.py"), os.path.join(build, "server.py"))

# 5. Start the server. The dot size is passed through a tiny wrapper script.
wrapper = os.path.join(build, "run_viewer.sh")
open(wrapper, "w").write(f'#!/bin/sh\nexec "{os.path.join(build, "viewer_real")}" "$1" {DOT_SIZE_M}\n')
os.chmod(wrapper, 0o755)
log = open(log_file, "w")
server = subprocess.Popen(
    [sys.executable, os.path.join(build, "server.py"), "--renderer", wrapper, "--scene", scene,
     "--port", str(PORT), "--host", "0.0.0.0"],
    stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
open(pid_file, "w").write(str(server.pid))

ready = False
for _ in range(240):                     # a big scene takes a while to load onto the GPU
    if server.poll() is not None:
        break
    try:
        ready = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/ping", timeout=1).read() == b"ok"
        if ready:
            break
    except OSError:
        time.sleep(0.5)
print(open(log_file).read())
if not ready:
    raise SystemExit("The viewer server did not start. The lines above are its log.")

# 6. One picture straight from the server, shown here, so there is something to see even if the page fails.
reply = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/frame?reset=1", timeout=30)
first = reply.read()
print("first picture:", len(first), "bytes,",
      "kernels", reply.headers.get("X-Kernel-Ms"), "ms,",
      "whole picture on GPU", reply.headers.get("X-Frame-Ms"), "ms,",
      reply.headers.get("X-Dots"), "dots")
display(Image(first))

# 7. The page. Click the picture, then fly: W A S D, E / Q up and down, arrows to turn, R to reset.
from google.colab import output
try:
    output.serve_kernel_port_as_window(PORT, anchor_text="Open the viewer in its own tab (use this if the box below stays empty)")
except Exception as problem:
    print("could not make the link:", problem)
try:
    output.serve_kernel_port_as_iframe(PORT, height="640")
except Exception as problem:
    print("could not show the page here:", problem)
